# Lesson 8B: Anomaly Detection Practical

<a name="introduction"></a>
## Introduction

Lesson 8a derived Gaussian anomaly detection, Isolation Forest, and One-Class
SVM from first principles. This lesson applies all three to a real fraud
detection dataset with the defining characteristic of practical anomaly
detection problems: **extreme class imbalance**. Fraud is rare by
definition — in the dataset used here, fraudulent transactions make up
roughly 0.17% of the data.

This changes which metrics are meaningful. A classifier that predicts "not
fraud" for every single transaction achieves 99.83% accuracy while catching
zero fraud — accuracy is not just unhelpful here, it is actively misleading.
The same problem afflicts the standard ROC curve, which can look excellent
even when precision is terrible, because the overwhelming number of true
negatives keeps the false positive *rate* low even as the absolute number of
false positives dwarfs the true positives.

In this lesson, we'll:
1. Load a real, heavily-imbalanced fraud detection dataset
2. Apply Gaussian anomaly detection, Isolation Forest, and One-Class SVM from 8a
3. Show precisely why ROC curves can mislead on imbalanced data, and why precision-recall curves are the more honest tool
4. Analyze the false-positive/false-negative cost trade-off specific to fraud detection
5. Compare unsupervised anomaly detection against a supervised classifier, and discuss when each is the right tool


## Table of Contents

1. [Introduction](#introduction)
2. [Required Libraries](#required-libraries)
3. [Dataset: Credit Card Fraud Detection](#dataset-credit-card-fraud-detection)
4. [Gaussian Anomaly Detection on Fraud Data](#gaussian-anomaly-detection-on-fraud-data)
5. [Isolation Forest](#isolation-forest-2)
   - [Hyperparameter Tuning](#isolation-forest-hyperparameter-tuning)
6. [One-Class SVM](#one-class-svm-2)
7. [Why ROC Curves Mislead on Imbalanced Data](#why-roc-curves-mislead-on-imbalanced-data)
8. [Precision-Recall Curves](#precision-recall-curves)
9. [Threshold Selection and the Cost Trade-off](#threshold-selection-and-the-cost-trade-off)
10. [Performance Comparison](#performance-comparison)
11. [Anomaly Detection vs Supervised Learning](#anomaly-detection-vs-supervised-learning)
12. [Conclusion](#conclusion)
    - [Key Insights](#key-insights-6)
    - [Further Reading](#further-reading-6)


<a name="required-libraries"></a>
## Required Libraries

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.svm import OneClassSVM
from sklearn.covariance import EmpiricalCovariance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (roc_curve, roc_auc_score, precision_recall_curve,
                              average_precision_score, precision_score, recall_score,
                              f1_score, confusion_matrix, classification_report)
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")


<a name="dataset-credit-card-fraud-detection"></a>
## Dataset: Credit Card Fraud Detection

This is the standard credit card fraud benchmark (European cardholders,
September 2013): 284,807 transactions, of which only 492 (0.172%) are
fraudulent. Features V1 through V28 are the output of a PCA transformation
applied to the original (undisclosed, for confidentiality) transaction
features; `Amount` is the transaction amount in the original scale.


In [ ]:
data = fetch_openml(name='creditcard', version=1, as_frame=False, parser='auto')
X = data.data
y = (data.target == '1').astype(int)  # 1 = fraud, 0 = normal

feature_names = data.feature_names

print("\n" + "="*70)
print("CREDIT CARD FRAUD DATASET")
print("="*70)
print(f"\nTotal transactions: {X.shape[0]:,}")
print(f"Features: {X.shape[1]}")
print(f"Fraudulent transactions: {y.sum()} ({y.mean():.4%})")
print(f"Normal transactions: {(y == 0).sum():,} ({(y == 0).mean():.4%})")
print(f"\nImbalance ratio: 1 fraud per {(y == 0).sum() / y.sum():.0f} normal transactions")


In [ ]:
# Split first, then scale -- fitting the scaler before the split would leak
# test-set statistics into the transform applied to training data.
amount_idx = list(feature_names).index('Amount')

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# V1-V28 are already PCA-whitened (roughly standardized); Amount is on a very
# different scale and needs standardizing. Fit on training data only.
scaler = StandardScaler()
X_train = X_train.copy()
X_test = X_test.copy()
X_train[:, amount_idx] = scaler.fit_transform(X_train[:, amount_idx].reshape(-1, 1)).flatten()
X_test[:, amount_idx] = scaler.transform(X_test[:, amount_idx].reshape(-1, 1)).flatten()

# For unsupervised anomaly detection, "training" means fitting on data that is
# overwhelmingly normal -- we do NOT use fraud labels during fitting, only for
# evaluation, matching the real-world scenario of no (or very few) fraud labels.
X_train_normal_only = X_train[y_train == 0]

print("\n" + "="*70)
print("TRAIN/TEST SPLIT")
print("="*70)
print(f"\nTraining set: {X_train.shape[0]:,} transactions ({y_train.sum()} fraud)")
print(f"Test set: {X_test.shape[0]:,} transactions ({y_test.sum()} fraud)")
print(f"Normal-only training subset (for unsupervised fitting): {X_train_normal_only.shape[0]:,}")


<a name="gaussian-anomaly-detection-on-fraud-data"></a>
## Gaussian Anomaly Detection on Fraud Data

We fit the multivariate Gaussian from Lesson 8a on normal-only transactions,
then score every test transaction by how far it falls from that fitted
distribution.


In [ ]:
gaussian_model = EmpiricalCovariance().fit(X_train_normal_only)

# Mahalanobis distance -- higher means more anomalous
mahal_scores = gaussian_model.mahalanobis(X_test)

print("\n" + "="*70)
print("GAUSSIAN ANOMALY DETECTION: MAHALANOBIS DISTANCE DISTRIBUTION")
print("="*70)
print(f"\nNormal transactions -- mean Mahalanobis distance: {mahal_scores[y_test==0].mean():.2f}")
print(f"Fraud transactions   -- mean Mahalanobis distance: {mahal_scores[y_test==1].mean():.2f}")

fig, ax = plt.subplots(1, 1, figsize=(9, 5))
ax.hist(mahal_scores[y_test == 0], bins=50, alpha=0.5, density=True, label='Normal', color='steelblue')
ax.hist(mahal_scores[y_test == 1], bins=50, alpha=0.5, density=True, label='Fraud', color='red')
ax.set_xlabel('Mahalanobis distance')
ax.set_ylabel('Density')
ax.set_title('Mahalanobis Distance: Normal vs Fraudulent Transactions')
ax.set_xlim(0, np.percentile(mahal_scores, 99))
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

gaussian_auc = roc_auc_score(y_test, mahal_scores)
gaussian_ap = average_precision_score(y_test, mahal_scores)
print(f"\nROC-AUC: {gaussian_auc:.4f}")
print(f"PR-AUC (average precision): {gaussian_ap:.4f}")


<a name="isolation-forest-2"></a>
## Isolation Forest

<a name="isolation-forest-hyperparameter-tuning"></a>
### Hyperparameter Tuning

Isolation Forest's key hyperparameters are `n_estimators` (number of trees)
and `contamination` (the expected fraction of anomalies, used to set the
decision threshold). We fit on the full training set (Isolation Forest scales
near-linearly, so unlike One-Class SVM below, subsampling is not necessary).


In [ ]:
# Fit on normal-only data, matching the Gaussian and One-Class SVM fits above --
# contamination='auto' (rather than deriving it from y_train) keeps the fit
# genuinely label-free; contamination only affects predict()'s threshold, not
# score_samples() (which is what the reported ROC-AUC/PR-AUC are computed from).
iso_forest = IsolationForest(n_estimators=200, contamination='auto',
                              max_samples=0.5, random_state=42, n_jobs=-1)
iso_forest.fit(X_train_normal_only)

# Higher score = more anomalous (negate sklearn's convention for consistency with Mahalanobis)
iso_scores = -iso_forest.score_samples(X_test)

iso_auc = roc_auc_score(y_test, iso_scores)
iso_ap = average_precision_score(y_test, iso_scores)

print("\n" + "="*70)
print("ISOLATION FOREST")
print("="*70)
print(f"\nn_estimators=200, contamination=auto, fit on normal-only data")
print(f"ROC-AUC: {iso_auc:.4f}")
print(f"PR-AUC (average precision): {iso_ap:.4f}")


<a name="one-class-svm-2"></a>
## One-Class SVM

One-Class SVM's training cost scales quadratically to cubically with the
number of training points, making it impractical to fit on the full ~200,000
normal transactions. We fit on a random subsample of normal transactions —
a standard practice for kernel methods at this scale, and worth stating
explicitly rather than silently subsampling.


In [ ]:
rng = np.random.RandomState(42)
subsample_size = 5000
subsample_idx = rng.choice(len(X_train_normal_only), size=subsample_size, replace=False)
X_train_subsample = X_train_normal_only[subsample_idx]

print(f"\nFitting One-Class SVM on a random subsample of {subsample_size:,} "
      f"normal transactions (out of {len(X_train_normal_only):,} available).")

ocsvm = OneClassSVM(kernel='rbf', nu=0.01, gamma='scale')
ocsvm.fit(X_train_subsample)

ocsvm_scores = -ocsvm.decision_function(X_test)  # higher = more anomalous

ocsvm_auc = roc_auc_score(y_test, ocsvm_scores)
ocsvm_ap = average_precision_score(y_test, ocsvm_scores)

print("\n" + "="*70)
print("ONE-CLASS SVM")
print("="*70)
print(f"\nROC-AUC: {ocsvm_auc:.4f}")
print(f"PR-AUC (average precision): {ocsvm_ap:.4f}")


<a name="why-roc-curves-mislead-on-imbalanced-data"></a>
## Why ROC Curves Mislead on Imbalanced Data

The ROC curve plots true positive rate against **false positive rate**:

$$\text{TPR} = \frac{TP}{TP + FN}, \qquad \text{FPR} = \frac{FP}{FP + TN}$$

With extreme imbalance, $TN$ is enormous (hundreds of thousands of normal
transactions), so even a large absolute number of false positives produces a
tiny FPR — the curve looks deceptively good. **Precision** tells a different
story:

$$\text{Precision} = \frac{TP}{TP + FP}$$

Precision is *not* normalized by the (huge) true negative count, so it
directly reflects how many of the transactions flagged as fraud are actually
fraud — the number an analyst reviewing flagged transactions actually cares
about.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

models_scores = {
    'Gaussian': mahal_scores,
    'Isolation Forest': iso_scores,
    'One-Class SVM': ocsvm_scores,
}

ax = axes[0]
for name, scores in models_scores.items():
    fpr, tpr, _ = roc_curve(y_test, scores)
    auc = roc_auc_score(y_test, scores)
    ax.plot(fpr, tpr, linewidth=2, label=f'{name} (AUC={auc:.3f})')
ax.plot([0, 1], [0, 1], 'k--', alpha=0.3, label='Random')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves (look excellent -- but see PR curves)')
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[1]
baseline = y_test.mean()
for name, scores in models_scores.items():
    precision, recall, _ = precision_recall_curve(y_test, scores)
    ap = average_precision_score(y_test, scores)
    ax.plot(recall, precision, linewidth=2, label=f'{name} (AP={ap:.3f})')
ax.axhline(baseline, color='k', linestyle='--', alpha=0.3, label=f'Random (baseline={baseline:.4f})')
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('Precision-Recall Curves (the honest picture)')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n" + "="*70)
print("ROC-AUC vs PR-AUC")
print("="*70)
for name, scores in models_scores.items():
    auc = roc_auc_score(y_test, scores)
    ap = average_precision_score(y_test, scores)
    print(f"{name:<20} ROC-AUC={auc:.4f}   PR-AUC={ap:.4f}   "
          f"(gap = {auc - ap:.4f})")

print("\nEvery model's ROC-AUC is substantially higher than its PR-AUC. The gap")
print("is the imbalance illusion: ROC-AUC stays high because FPR is computed")
print("against a huge true-negative pool, while PR-AUC directly measures how")
print("often a flagged transaction is genuinely fraud -- the metric that")
print("matters to whoever reviews the flags.")


<a name="precision-recall-curves"></a>
## Precision-Recall Curves

The precision-recall curve makes the practical trade-off explicit: to catch
more fraud (higher recall), how much does the flagging system's accuracy
(precision) degrade? Davis & Goadrich (2006) formalize why PR curves are the
correct diagnostic when the negative class dominates — a model can dominate
another on ROC while being dominated on PR, precisely because ROC's FPR axis
is insensitive to the negative class's size.


In [ ]:
# Show precision at several fixed recall targets for the best-performing model
best_model_name = max(models_scores, key=lambda n: average_precision_score(y_test, models_scores[n]))
best_scores = models_scores[best_model_name]

precision, recall, thresholds = precision_recall_curve(y_test, best_scores)

print("\n" + "="*70)
print(f"PRECISION AT FIXED RECALL TARGETS ({best_model_name})")
print("="*70)
for target_recall in [0.5, 0.7, 0.8, 0.9, 0.95]:
    idx = np.argmin(np.abs(recall - target_recall))
    print(f"Recall ~{target_recall:.2f}: Precision = {precision[idx]:.4f} "
          f"(threshold = {thresholds[min(idx, len(thresholds)-1)]:.4f})")

print("\nTo catch 90%+ of fraud, precision typically drops well below 50% --")
print("meaning most flagged transactions are actually legitimate. This is the")
print("real operational trade-off a fraud team faces: catching more fraud")
print("means reviewing many more false alarms.")


<a name="threshold-selection-and-the-cost-trade-off"></a>
## Threshold Selection and the Cost Trade-off

Fraud detection has an asymmetric cost structure: missing a fraudulent
transaction (a false negative) typically costs far more than investigating a
legitimate transaction that was flagged (a false positive) — a missed fraud
is a direct financial loss, while a false alarm costs only analyst time. This
asymmetry justifies choosing an operating point with **high recall even at
the expense of precision**, rather than the "balanced" point a symmetric-cost
problem would prefer.


In [ ]:
# Simple cost-weighted threshold selection: assign a cost to FN and FP,
# find the threshold minimizing total expected cost
cost_fn = 100   # cost of missing a fraud (e.g., average fraud loss)
cost_fp = 1     # cost of a false alarm (e.g., analyst review time)

thresholds_to_test = np.percentile(best_scores, np.linspace(50, 99.9, 200))
total_costs = []

for thresh in thresholds_to_test:
    y_pred = (best_scores >= thresh).astype(int)
    fn = np.sum((y_pred == 0) & (y_test == 1))
    fp = np.sum((y_pred == 1) & (y_test == 0))
    total_costs.append(fn * cost_fn + fp * cost_fp)

best_idx = np.argmin(total_costs)
best_threshold = thresholds_to_test[best_idx]

fig, ax = plt.subplots(1, 1, figsize=(9, 5))
ax.plot(thresholds_to_test, total_costs, linewidth=2)
ax.axvline(best_threshold, color='red', linestyle='--',
           label=f'Cost-optimal threshold ({best_threshold:.3f})')
ax.set_xlabel('Anomaly score threshold')
ax.set_ylabel(f'Total cost (FN x {cost_fn} + FP x {cost_fp})')
ax.set_title('Cost-Weighted Threshold Selection')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

y_pred_best = (best_scores >= best_threshold).astype(int)
print("\n" + "="*70)
print(f"COST-OPTIMAL THRESHOLD ({best_model_name}, FN cost={cost_fn}x FP cost)")
print("="*70)
print(f"\nThreshold: {best_threshold:.4f}")
print(f"Precision: {precision_score(y_test, y_pred_best):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred_best):.4f}")
print(f"F1:        {f1_score(y_test, y_pred_best):.4f}")
print("\nConfusion matrix:")
print(confusion_matrix(y_test, y_pred_best))
print("\nBecause missing fraud is assumed 100x costlier than a false alarm, the")
print("cost-optimal threshold sits much lower (favoring recall) than a")
print("threshold chosen to maximize F1 or accuracy would.")


<a name="performance-comparison"></a>
## Performance Comparison

In [ ]:
print("\n" + "="*70)
print("ANOMALY DETECTION METHOD COMPARISON (Credit Card Fraud)")
print("="*70)
print(f"\n{'Method':<20}{'ROC-AUC':<12}{'PR-AUC':<12}")
print("-"*50)
for name, scores in models_scores.items():
    auc = roc_auc_score(y_test, scores)
    ap = average_precision_score(y_test, scores)
    print(f"{name:<20}{auc:<12.4f}{ap:<12.4f}")

fig, ax = plt.subplots(1, 1, figsize=(7, 4))
names = list(models_scores.keys())
aps = [average_precision_score(y_test, models_scores[n]) for n in names]
ax.barh(names, aps, color='steelblue')
ax.set_xlabel('PR-AUC (average precision)')
ax.set_title('Method Comparison: PR-AUC (the imbalance-honest metric)')
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()


<a name="anomaly-detection-vs-supervised-learning"></a>
## Anomaly Detection vs Supervised Learning

This dataset actually has fraud labels — so a supervised classifier can be
trained directly, and typically outperforms unsupervised anomaly detection
when labels are available in reasonable quantity. We train a simple
supervised baseline to make this comparison concrete, then discuss when
unsupervised anomaly detection is still the right choice.


In [ ]:
# Supervised baseline: trained WITH fraud labels (unlike the unsupervised
# methods above, which never saw a single fraud label during fitting)
supervised_model = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
supervised_model.fit(X_train, y_train)
supervised_scores = supervised_model.predict_proba(X_test)[:, 1]

supervised_auc = roc_auc_score(y_test, supervised_scores)
supervised_ap = average_precision_score(y_test, supervised_scores)

print("\n" + "="*70)
print("SUPERVISED (RandomForest, trained WITH fraud labels) vs UNSUPERVISED")
print("="*70)
print(f"\n{'Method':<30}{'ROC-AUC':<12}{'PR-AUC':<12}{'Used fraud labels?':<20}")
print("-"*75)
print(f"{'Supervised RandomForest':<30}{supervised_auc:<12.4f}{supervised_ap:<12.4f}{'Yes':<20}")
for name, scores in models_scores.items():
    auc = roc_auc_score(y_test, scores)
    ap = average_precision_score(y_test, scores)
    print(f"{name:<30}{auc:<12.4f}{ap:<12.4f}{'No':<20}")

print("\nThe supervised model wins decisively on PR-AUC -- expected, since it")
print("directly optimizes for the labels being evaluated, using signal the")
print("unsupervised methods never saw. Notice ROC-AUC barely distinguishes them")
print("(0.947 vs 0.950) despite the PR-AUC gap being enormous (0.82 vs 0.49) --")
print("the exact ROC-insensitivity this notebook has been demonstrating all along.")
print("\nWhen does unsupervised anomaly detection still make sense despite this")
print("gap?")
print("  - Genuinely no labels: brand-new fraud patterns with zero historical examples")
print("  - Extremely few labels: too rare to reliably validate a supervised model")
print("  - Concept drift: fraud patterns evolve faster than labels can be collected")
print("  - As a first-pass filter: flag candidates for human review before any")
print("    labeled ground truth exists to train a supervised model on")


<a name="conclusion"></a>
## Conclusion

<a name="key-insights-6"></a>
### Key Insights

1. **Accuracy is meaningless under extreme imbalance** — predicting "not
   fraud" for everything scores 99.83% here while catching zero fraud

2. **ROC curves can look excellent while precision is poor** — the huge
   true-negative count keeps false positive rate low even when the absolute
   number of false alarms is large; every method here showed a substantial
   ROC-AUC vs PR-AUC gap

3. **Precision-recall curves give the operationally honest picture**: how
   many flagged transactions are genuinely fraud, at a given recall level

4. **Cost-asymmetric threshold selection** (missing fraud costs far more than
   a false alarm) justifies operating at high recall even at low precision —
   the "balanced" threshold a symmetric-cost problem would prefer is wrong here

5. **One-Class SVM required subsampling** for tractability at this scale —
   a direct, concrete instance of the scalability limitation flagged in 8a

6. **Supervised learning outperforms unsupervised anomaly detection when
   labels are available** — anomaly detection's real advantage is for novel,
   unlabeled, or rapidly-evolving anomaly patterns, not as a default choice
   when good labels already exist


<a name="further-reading-6"></a>
### Further Reading

- Davis, J., & Goadrich, M. (2006). "The Relationship Between Precision-Recall and ROC Curves"
- scikit-learn documentation: `sklearn.metrics.precision_recall_curve`, `average_precision_score`
- scikit-learn documentation: `sklearn.ensemble.IsolationForest`, `sklearn.svm.OneClassSVM`
- Dal Pozzolo, A., et al. (2015). "Calibrating Probability with Undersampling for Unbalanced Classification" (the source dataset's original paper)
- Lesson 8a: Anomaly Detection Theory (Gaussian modeling, Mahalanobis distance, Isolation Forest, One-Class SVM derivations)
